# Chapter 1 - Installation, setup and the Bohol dataset

Welcome to the **PyPSA bootcamp**. The whole course is built around one real
system: the electricity island of **Bohol** (Philippines). Every chapter adds
one new modelling idea on top of the same data, exactly the way you would learn
a programming language one concept at a time.

PyPSA (*Python for Power System Analysis*) is a free library that lets you:

- build a power network (buses, lines, generators, loads, storage) as a set of tables,
- run a **power flow** (how the current actually distributes in the grid),
- run an **optimisation** (how to operate or invest in the system at least cost).

**Goals of this chapter**
- Check that PyPSA and a linear solver (HiGHS) are installed and working.
- Load the Bohol dataset and look at what is inside it.
- Understand the eight component types we will use for the rest of the bootcamp.

### 1.1 Check the installation

If the imports below run without error, you are ready. `pypsa.__version__`
should be 1.x. We also silence a few harmless warnings so the output stays clean.

In [1]:
import warnings, logging
warnings.filterwarnings("ignore")
for _n in ("pypsa", "linopy", "pypsa.consistency"):
    logging.getLogger(_n).setLevel(logging.ERROR)

from pathlib import Path
import numpy as np
import pandas as pd
import pypsa

print("PyPSA   :", pypsa.__version__)
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)

PyPSA   : 1.4.0
pandas  : 3.1.1
numpy   : 2.4.6


### 1.2 Check the solver

Optimisation chapters (4 to 8) need a Linear Programming solver. PyPSA ships
with **HiGHS**, which is open source and needs no license. We test it on a
trivial one-line problem: one 100 MW generator feeding one 80 MW load. The
only possible answer is "produce 80 MW", at a cost of 80 x 10 = 800.

In [2]:
test = pypsa.Network()
test.add("Bus", "b")
test.add("Generator", "g", bus="b", p_nom=100, marginal_cost=10)
test.add("Load", "d", bus="b", p_set=80)

status, condition = test.optimize(solver_name="highs",
                                  solver_options={"output_flag": False},
                                  log_to_console=False)
print("solver status :", status, "/", condition)
print("generator output :", round(test.generators_t.p.iloc[0, 0], 1), "MW")
print("total cost       :", round(float(test.objective), 1), "currency units")
assert status == "ok", "HiGHS is not available - check the installation"
print("\nSolver works. You are ready for the whole bootcamp.")

solver status : ok / optimal
generator output : 80.0 MW
total cost       : 800.0 currency units

Solver works. You are ready for the whole bootcamp.


### 1.3 Locate the Bohol dataset

The dataset is a single Excel file with seven sheets. We find it whether the
notebook is run from the project root or from the `notebooks/` folder.

In [3]:
DATA = next(p for p in [Path("data"), Path("../data"), Path("pypsa-bootcamp/data")]
            if (p / "PyPSA_Dataset_Bohol.xlsx").exists())
XLS = DATA / "PyPSA_Dataset_Bohol.xlsx"
sheets = pd.ExcelFile(XLS).sheet_names
print("dataset folder :", DATA.resolve())
print("sheets         :", sheets)

dataset folder : /home/user/piggy-cast/pypsa-bootcamp/data
sheets         : ['buses', 'loads', 'carriers', 'generators', 'lines', 'loads_t', 'generators_t']


### 1.4 What is in each sheet

The seven sheets map one-to-one onto PyPSA components. This is the vocabulary
of the whole bootcamp, so take a minute here.

| Sheet | PyPSA component | Meaning |
| --- | --- | --- |
| `buses` | **Bus** | an electrical node (a substation), at a voltage level |
| `lines` | **Line** | an AC transmission line joining two buses |
| `loads` | **Load** | electricity demand connected to a bus |
| `generators` | **Generator** | a power plant (diesel, hydro, grid import) |
| `carriers` | **Carrier** | a technology/fuel, with its cost and CO2 content |
| `loads_t` | time series | hourly demand for one day (the `_t` means "time dependent") |
| `generators_t` | time series | hourly availability of solar and wind |

Let us look at the static sheets.

In [4]:
buses = pd.read_excel(XLS, "buses")
print("BUSES - the three nodes of the Bohol grid")
print(buses.to_string(index=False))

BUSES - the three nodes of the Bohol grid
     name          x         y  v_nom
07CORELLA 123.959325  9.699520  138.0
  07TAPAL 124.519576 10.060901   13.8
   07UBAY 124.511428 10.026670  138.0


Three buses: **07CORELLA** and **07UBAY** at 138 kV (the transmission backbone)
and **07TAPAL** at 13.8 kV (a distribution pocket). The `x` and `y` columns are
the real longitude/latitude, which we will use to draw the network on a map.

In [5]:
lines = pd.read_excel(XLS, "lines")
print("LINES - the AC corridors")
print(lines.to_string(index=False))
print("\nGENERATORS - the power plants")
print(pd.read_excel(XLS, "generators").to_string(index=False))

LINES - the AC corridors
                name  cables   bus0      bus1  s_nom         x        r
   69_07UBAY_07TAPAL       1 07UBAY   07TAPAL     69  1.875327 0.305012
138_07UBAY_07CORELLA       1 07UBAY 07CORELLA    138 12.089197 8.512958
 69_07UBAY_07CORELLA       1 07UBAY 07CORELLA     69 33.857589 5.506756

GENERATORS - the power plants


            name       bus  p_nom carrier  build_year
    07TPLPB4_U01   07TAPAL    7.0  Diesel        2005
    07TPLPB4_U02   07TAPAL    7.0  Diesel        2005
    07TPLPB4_U03   07TAPAL    7.0  Diesel        2005
    07TPLPB4_U04   07TAPAL    8.0  Diesel        2005
      07BDPP_U01 07CORELLA    4.0  Diesel        1978
      07BDPP_U02 07CORELLA    4.0  Diesel        1978
      07BDPP_U03 07CORELLA    4.2  Diesel        1978
      07BDPP_U04 07CORELLA    4.0  Diesel        1978
    07JANOPO_G01 07CORELLA    5.0   Hydro        1992
     07LOBOC_G01 07CORELLA    1.2   Hydro        2019
     07LOBOC_G02 07CORELLA    1.2   Hydro        2019
    07SEVILL_G01 07CORELLA    2.5   Hydro        2008
Grid_electricity    07UBAY  138.0    Grid        2005


Thirteen generators: four diesel units at Tapal, four older diesel units and
four small hydro units at Corella, plus one "Grid" generator at Ubay that stands
for the submarine cable importing power from the neighbouring island of Leyte.

### 1.5 Carriers: where the economics live

A **Carrier** is a fuel or technology. PyPSA uses it to attach two things to
every plant of that type: its **CO2 content** and the raw numbers we turn into
a **marginal cost** (currency per MWh produced) and a **capital cost** (currency
per MW of capacity). We will compute those in Chapter 4; for now just read them.

In [6]:
carriers = pd.read_excel(XLS, "carriers")
print("CARRIERS")
print(carriers.to_string(index=False))

CARRIERS
   name   color  co2_emissions    VOM    FOM     fuel   investment  efficiency  lifetime
 Diesel #B41E25         0.2571 6.3493 2.5656 52.91110 3.629700e+05        0.35        25
  Hydro #1E81D4         0.0000 0.0000 1.0000  0.00000 2.274818e+06        0.90        80
  Solar #FDB324         0.0000 0.0106 1.5780  0.00000 8.098118e+05        1.00        35
   Wind #3BBFE5         0.0000 1.5873 1.2514  0.00000 1.183912e+06        1.00        27
Battery #814ad4         0.0000 0.0000 0.2000  0.00000 2.455074e+05        0.95        20
   Grid #b01787         0.1100 0.0000 0.0000 42.73002 1.396283e+04        1.00        40


Reading the economics the way the rest of the bootcamp will:

- **marginal cost** = `VOM + fuel / efficiency` (variable O&M plus the fuel
  burned per MWh of electricity). Diesel burns expensive fuel at 35 % efficiency,
  so it will be costly; hydro and the grid import are cheap.
- **capital cost** = `(1 + FOM) * investment / (lifetime * 365.25)`, i.e. the
  build cost spread over the plant's life and expressed **per day** (our
  optimisation horizon is one day).

In [7]:
econ = carriers.assign(
    marginal_cost=lambda d: (d.VOM + d.fuel / d.efficiency).round(2),
    capital_cost_per_day=lambda d: ((1 + d.FOM) * d.investment / (d.lifetime * 365.25)).round(1),
)[["name", "co2_emissions", "marginal_cost", "capital_cost_per_day"]]
print(econ.to_string(index=False))

   name  co2_emissions  marginal_cost  capital_cost_per_day
 Diesel         0.2571         157.52                 141.7
  Hydro         0.0000           0.00                 155.7
  Solar         0.0000           0.01                 163.3
   Wind         0.0000           1.59                 270.3
Battery         0.0000           0.00                  40.3
   Grid         0.1100          42.73                   1.0


Already a result worth noting: **diesel costs about 157 /MWh, the grid import
about 43 /MWh, and hydro is almost free.** Any least-cost model will lean on
hydro and the cable first and keep diesel for the peak. We will prove it in
Chapter 4.

### 1.6 The time series

The `_t` sheets hold one winter day (2 January 2023) at hourly resolution.

In [8]:
loads_t = pd.read_excel(XLS, "loads_t", index_col=0)
gens_t = pd.read_excel(XLS, "generators_t", index_col=0)
print("Demand (MW) - first hours:")
print(loads_t.head(3).round(2).to_string())
print("\nRenewable availability (fraction of capacity) - midday hours:")
print(gens_t.iloc[9:13].round(3).to_string())
print("\nDaily peak demand:", round(loads_t.sum(axis=1).max(), 1), "MW",
      "| daily energy:", round(loads_t.sum(axis=1).sum(), 0), "MWh")

Demand (MW) - first hours:
                     07CORELLA_LOAD  07UBAY_LOAD
2023-01-02 00:00:00           38.34        19.89
2023-01-02 01:00:00           35.27        18.58
2023-01-02 02:00:00           33.06        17.44

Renewable availability (fraction of capacity) - midday hours:
                     SOLAR_BOHOL  WIND_BOHOL
2023-01-02 09:00:00        0.589       0.842
2023-01-02 10:00:00        0.656       0.854
2023-01-02 11:00:00        0.666       0.855
2023-01-02 12:00:00        0.620       0.844

Daily peak demand: 82.6 MW | daily energy: 1516.0 MWh


Solar peaks at noon (~0.67 of its capacity) and is zero at night; wind is
strong all day on this date (0.4 to 0.85). The island peaks near 84 MW.

## Summary of Chapter 1

- PyPSA and the HiGHS solver are installed and solve a trivial problem correctly.
- The Bohol dataset has **3 buses, 3 lines, 13 generators, 6 carriers** and a
  24-hour demand and renewable profile.
- Economics come from the carriers: **diesel ~157, grid ~43, hydro ~0** per MWh.
- Next chapter we build our very first network by hand and read its tables.